In [16]:
pip install python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [19]:
import asyncio
import requests
import psycopg2
import os

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
OLLAMA_MODEL    = os.getenv("OLLAMA_MODEL")

PG_HOST     = os.getenv("PG_HOST")
PG_PORT     = os.getenv("PG_PORT")
PG_DB       = os.getenv("PG_DB")
PG_USER     = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

print("Config loaded successfully")

Config loaded successfully


In [4]:
def get_connection():
    return psycopg2.connect(
        host=PG_HOST, port=PG_PORT,
        dbname=PG_DB, user=PG_USER, password=PG_PASSWORD
    )

def execute_ddl(sql: str) -> str:
    try:
        conn = get_connection()
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute(sql)
        cur.close()
        conn.close()
        return "SUCCESS"
    except Exception as e:
        return f"ERROR: {str(e)}"

test = execute_ddl("SELECT 1")
print("DB connection:", test)

DB connection: SUCCESS


In [5]:
import re
from autogen_core.models import ModelInfo, CreateResult, RequestUsage
from autogen_core import CancellationToken

In [6]:
import re
from autogen_core.models import ModelInfo, CreateResult, RequestUsage
from autogen_core import CancellationToken

class OllamaChatClient:
    def __init__(self, model: str = OLLAMA_MODEL, base_url: str = OLLAMA_BASE_URL):
        self._model    = model
        self._endpoint = f"{base_url.rstrip('/')}/v1/chat/completions"
        self.model_info = ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False,
        )

    @property
    def capabilities(self):
        return self.model_info

    async def create(self, messages, **kwargs) -> CreateResult:
        # Build message list for Ollama
        ollama_messages = []
        for m in messages:
            if hasattr(m, "content"):
                content = m.content
                if isinstance(content, list):
                    content = " ".join(
                        item.text for item in content if hasattr(item, "text")
                    )
                ollama_messages.append({"role": "user", "content": content})

        response = requests.post(
            self._endpoint,
            json={
                "model":       self._model,
                "messages":    ollama_messages,
                "temperature": 0,
                "stream":      False,
            },
            timeout=120      # local inference can be slow on CPU
        )
        response.raise_for_status()

        text = response.json()["choices"][0]["message"]["content"].strip()

        # Strip markdown fences llama adds
        text = re.sub(r"^```[a-z]*\n?", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\n?```$",        "", text).strip()

        return CreateResult(
            content=text,
            usage=RequestUsage(prompt_tokens=0, completion_tokens=0),
            finish_reason="stop",
            cached=False,
        )

    def count_tokens(self, messages, **kwargs):
        return 0

    def remaining_tokens(self, messages, **kwargs):
        return 10000


class ExecutorChatClient:
    def __init__(self):
        self.model_info = ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False,
        )

    @property
    def capabilities(self):
        return self.model_info

    async def create(self, messages, **kwargs) -> CreateResult:
        ddl = ""
        for m in reversed(messages):
            if hasattr(m, "content") and isinstance(m.content, str):
                ddl = m.content
                break
        result = execute_ddl(ddl)
        return CreateResult(
            content=result,
            usage=RequestUsage(prompt_tokens=0, completion_tokens=0),
            finish_reason="stop",
            cached=False,
        )

    def count_tokens(self, messages, **kwargs):
        return 0

    def remaining_tokens(self, messages, **kwargs):
        return 10000


# Sanity check — verify Ollama is reachable and model is pulled
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        if not any(OLLAMA_MODEL.split(":")[0] in m for m in models):
            print(f"⚠ Model '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
            print(f"  Available models: {models}")
        else:
            print(f"✅ Ollama running — model '{OLLAMA_MODEL}' ready")
    except requests.exceptions.ConnectionError:
        print("❌ Cannot reach Ollama. Run: ollama serve")

check_ollama()
print("Clients ready")

✅ Ollama running — model 'llama3.2' ready
Clients ready


In [7]:
class OllamaChatClient:
    def __init__(self, model: str = OLLAMA_MODEL, base_url: str = OLLAMA_BASE_URL):
        self._model    = model
        self._endpoint = f"{base_url.rstrip('/')}/v1/chat/completions"
        self.model_info = ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False,
        )

    @property
    def capabilities(self):
        return self.model_info

    async def create(self, messages, **kwargs) -> CreateResult:
        # Build message list for Ollama
        ollama_messages = []
        for m in messages:
            if hasattr(m, "content"):
                content = m.content
                if isinstance(content, list):
                    content = " ".join(
                        item.text for item in content if hasattr(item, "text")
                    )
                ollama_messages.append({"role": "user", "content": content})

        response = requests.post(
            self._endpoint,
            json={
                "model":       self._model,
                "messages":    ollama_messages,
                "temperature": 0,
                "stream":      False,
            },
            timeout=120      # local inference can be slow on CPU
        )
        response.raise_for_status()

        text = response.json()["choices"][0]["message"]["content"].strip()

        # Strip markdown fences llama adds
        text = re.sub(r"^```[a-z]*\n?", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\n?```$",        "", text).strip()

        return CreateResult(
            content=text,
            usage=RequestUsage(prompt_tokens=0, completion_tokens=0),
            finish_reason="stop",
            cached=False,
        )

    def count_tokens(self, messages, **kwargs):
        return 0

    def remaining_tokens(self, messages, **kwargs):
        return 10000


class ExecutorChatClient:
    def __init__(self):
        self.model_info = ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family="unknown",
            structured_output=False,
        )

    @property
    def capabilities(self):
        return self.model_info

    async def create(self, messages, **kwargs) -> CreateResult:
        ddl = ""
        for m in reversed(messages):
            if hasattr(m, "content") and isinstance(m.content, str):
                ddl = m.content
                break
        result = execute_ddl(ddl)
        return CreateResult(
            content=result,
            usage=RequestUsage(prompt_tokens=0, completion_tokens=0),
            finish_reason="stop",
            cached=False,
        )

    def count_tokens(self, messages, **kwargs):
        return 0

    def remaining_tokens(self, messages, **kwargs):
        return 10000


# Sanity check — verify Ollama is reachable and model is pulled
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
        if not any(OLLAMA_MODEL.split(":")[0] in m for m in models):
            print(f"⚠ Model '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
            print(f"  Available models: {models}")
        else:
            print(f"✅ Ollama running — model '{OLLAMA_MODEL}' ready")
    except requests.exceptions.ConnectionError:
        print("❌ Cannot reach Ollama. Run: ollama serve")

check_ollama()
print("Clients ready")

✅ Ollama running — model 'llama3.2' ready
Clients ready


In [8]:
from autogen_agentchat.agents import AssistantAgent

translator = AssistantAgent(
    name="Translator",
    model_client=OllamaChatClient(),
    system_message="""
You are a SQL Server to PostgreSQL Migration expert.
Convert the given SQL Server DDL into valid PostgreSQL DDL.
Rules:
- Replace IDENTITY(1,1) with SERIAL or GENERATED ALWAYS AS IDENTITY
- Replace NVARCHAR(n) with VARCHAR(n)
- Replace DATETIME with TIMESTAMP
- Replace GETDATE() with NOW()
- Replace MONEY with NUMERIC(19,4)
- Replace BIT with BOOLEAN, and default 1/0 with TRUE/FALSE
- Remove any square brackets [ ]
Only return the converted SQL code. Provide no explanations.
""",
)

executor = AssistantAgent(
    name="Executor",
    model_client=ExecutorChatClient(),
    system_message="Execute the given PostgreSQL DDL and return SUCCESS or FAILURE.",
)

self_corrector = AssistantAgent(
    name="SelfCorrector",
    model_client=OllamaChatClient(),
    system_message="""
You are a PostgreSQL DDL expert. Fix the provided DDL based on the error message.
Only return the corrected SQL code. Provide no explanations.
""",
)

validator = AssistantAgent(
    name="Validator",
    model_client=OllamaChatClient(),
    system_message="""
You are a PostgreSQL DDL reviewer.
Check if the given PostgreSQL DDL is correct and complete.
Reply with exactly VALID or INVALID: <reason>.
""",
)

print("Agents ready")

Agents ready


In [9]:
async def run_migration(sql_server_ddl: str, max_retries: int = 3) -> str | None:
    print("=" * 55)
    print("  SQL Server → PostgreSQL Migration Pipeline")
    print("=" * 55)

    # STEP 1: Translate 
    print("\n[1] Translating DDL...")
    result = await translator.run(task=sql_server_ddl)
    pg_ddl = result.messages[-1].content
    print(f"\nTranslated DDL:\n{pg_ddl}\n")

    current_ddl = pg_ddl

    # STEP 2: Execute (with self-correction retry loop)
    for attempt in range(1, max_retries + 1):
        print(f"[2] Executing DDL (attempt {attempt}/{max_retries})...")
        result = await executor.run(task=current_ddl)
        exec_result = result.messages[-1].content
        print(f"    Result: {exec_result}\n")

        if exec_result.startswith("SUCCESS"):
            break

        if attempt == max_retries:
            print("Migration FAILED - max retries reached.")
            return None

        # Route to self_corrector
        print(f"  Execution failed → routing to SelfCorrector...\n")
        fix_prompt = (
            f"The following PostgreSQL DDL failed:\n\n"
            f"{current_ddl}\n\n"
            f"Error:\n{exec_result}\n\n"
            f"Return only the corrected DDL."
        )
        result = await self_corrector.run(task=fix_prompt)
        current_ddl = result.messages[-1].content
        print(f"    Corrected DDL:\n{current_ddl}\n")

    # STEP 3: Validate 
    print("[3] Validating final DDL...")
    result = await validator.run(task=current_ddl)
    validation = result.messages[-1].content
    print(f"    Validator says: {validation}\n")

    if validation.startswith("VALID"):
        print("Migration COMPLETE!")
        return current_ddl
    else:
        print(f"Validation failed: {validation}")
        return None

In [ ]:
# sample_sql_server_ddl = """
# CREATE TABLE Employees (
#     EmployeeID   INT           IDENTITY(1,1) PRIMARY KEY,
#     FirstName    NVARCHAR(50)  NOT NULL,
#     LastName     NVARCHAR(50)  NOT NULL,
#     HireDate     DATETIME      DEFAULT GETDATE(),
#     Salary       MONEY,
#     IsActive     BIT           DEFAULT 1
# );

# CREATE TABLE Departments (
#     DeptID       INT           IDENTITY(1,1) PRIMARY KEY,
#     DeptName     NVARCHAR(100) NOT NULL,
#     ManagerID    INT,
#     FOREIGN KEY (ManagerID) REFERENCES Employees(EmployeeID)
# );
# """

# final_ddl = await run_migration(sample_sql_server_ddl)

In [10]:
cleanup = """
DROP TABLE IF EXISTS OrderItems CASCADE;
DROP TABLE IF EXISTS CustomerOrders CASCADE;
DROP TABLE IF EXISTS StoreProducts CASCADE;
DROP TABLE IF EXISTS StoreCategories CASCADE;
DROP TABLE IF EXISTS StaffMembers CASCADE;
DROP TABLE IF EXISTS StaffDepartments CASCADE;
DROP TABLE IF EXISTS InventoryItems CASCADE;
DROP TABLE IF EXISTS PayrollRecords CASCADE;
DROP TABLE IF EXISTS AuditTrail CASCADE;
DROP TABLE IF EXISTS UserAccounts CASCADE;
DROP TABLE IF EXISTS BrokenRefTable CASCADE;
DROP TABLE IF EXISTS BadCheckTable CASCADE;
DROP TABLE IF EXISTS DuplicateTarget CASCADE;
DROP TABLE IF EXISTS ResidualSyntaxTable CASCADE;
DROP TABLE IF EXISTS GhostRefTable CASCADE;
DROP TABLE IF EXISTS Products CASCADE;
"""
print(execute_ddl(cleanup))

SUCCESS


In [11]:
# TRANSLATION TEST CASE: TO CHECK IF THE MAJOR SQL DATA TYPES ARE CONVERTED PROPERLY
test_T1 = """
CREATE TABLE StaffMembers (
    StaffID       INT           IDENTITY(1,1) PRIMARY KEY,
    FirstName     NVARCHAR(50)  NOT NULL,
    LastName      NVARCHAR(50)  NOT NULL,
    JoinDate      DATETIME      DEFAULT GETDATE(),
    MonthlySalary MONEY         NOT NULL,
    IsActive      BIT           DEFAULT 1,
    Notes         NVARCHAR(MAX)
);
"""
print("\n>>> TEST T1: All major SQL Server data types")
result_T1 = await run_migration(test_T1)
print("\nFinal DDL:\n", result_T1)


>>> TEST T1: All major SQL Server data types
  SQL Server → PostgreSQL Migration Pipeline

[1] Translating DDL...

Translated DDL:
CREATE TABLE staff_members (
    staff_id SERIAL PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    join_date TIMESTAMP DEFAULT NOW(),
    monthly_salary NUMERIC(19,4) NOT NULL,
    is_active BOOLEAN DEFAULT TRUE,
    notes TEXT
);

[2] Executing DDL (attempt 1/3)...
    Result: SUCCESS

[3] Validating final DDL...
    Validator says: VALID 

The given DDL creates a table named 'staff_members' with the specified columns and constraints. The primary key is correctly defined as 'staff_id', which is also set to auto-increment using SERIAL data type. All other columns have their respective data types, constraints, and default values.

Migration COMPLETE!

Final DDL:
 CREATE TABLE staff_members (
    staff_id SERIAL PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    join_date TIMESTAM

In [ ]:
# EXECUTION TEST CASE: TO CHECK IF THE EXECUTOR CAN RUN A SIMPLE DDL WITHOUT ERRORS

test_E1 = """
CREATE TABLE UserAccounts (
    AccountID    INT           IDENTITY(1,1) PRIMARY KEY,
    Username     NVARCHAR(50)  NOT NULL,
    Email        NVARCHAR(100) NOT NULL,
    CreatedAt    DATETIME      DEFAULT GETDATE(),
    IsPremium    BIT           DEFAULT 0
);
"""
print("\n>>> TEST E1: Clean execution — expect SUCCESS on first attempt")
result_E1 = await run_migration(test_E1)


>>> TEST E1: Clean execution — expect SUCCESS on first attempt
  SQL Server → PostgreSQL Migration Pipeline

[1] Translating DDL...

Translated DDL:
CREATE TABLE UserAccounts (
    AccountID    INT           GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    Username     VARCHAR(50)   NOT NULL,
    Email        VARCHAR(100)  NOT NULL,
    CreatedAt    TIMESTAMP     DEFAULT NOW(),
    IsPremium    BOOLEAN       DEFAULT FALSE
);

[2] Executing DDL (attempt 1/3)...
    Result: SUCCESS

[3] Validating final DDL...
    Validator says: VALID

Migration COMPLETE!


In [12]:
# SELF-CORRECTION TEST CASE: TO CHECK IF THE SELF-CORRECTOR CAN FIX A DUPLICATE TABLE ERROR

execute_ddl("""
CREATE TABLE DuplicateTarget (
    ItemID SERIAL PRIMARY KEY,
    ItemName VARCHAR(100)
);
""")

test_SC1 = """
CREATE TABLE DuplicateTarget (
    ItemID      INT           IDENTITY(1,1) PRIMARY KEY,
    ItemName    NVARCHAR(100) NOT NULL,
    ItemPrice   MONEY,
    IsAvailable BIT DEFAULT 1
);
"""
print("\n>>> TEST SC1: Duplicate table")
result_SC1 = await run_migration(test_SC1)


>>> TEST SC1: Duplicate table
  SQL Server → PostgreSQL Migration Pipeline

[1] Translating DDL...

Translated DDL:
CREATE TABLE StaffMembers (
    StaffID       INT           GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    FirstName     VARCHAR(50)  NOT NULL,
    LastName      VARCHAR(50)  NOT NULL,
    JoinDate      TIMESTAMP DEFAULT NOW(),
    MonthlySalary NUMERIC(19,4) NOT NULL,
    IsActive      BOOLEAN DEFAULT TRUE,
    Notes         TEXT
);

CREATE TABLE staff_members (
    staff_id SERIAL PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    join_date TIMESTAMP DEFAULT NOW(),
    monthly_salary NUMERIC(19,4) NOT NULL,
    is_active BOOLEAN DEFAULT TRUE,
    notes TEXT
);

CREATE TABLE DuplicateTarget (
    ItemID      INT           GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    ItemName    VARCHAR(100) NOT NULL,
    ItemPrice   NUMERIC(19,4),
    IsAvailable BOOLEAN DEFAULT TRUE
);

[2] Executing DDL (attempt 1/3)...
    Result: ERROR: relatio

In [13]:
# VALIDATION TEST CASE: TO CHECK IF THE VALIDATOR CAN DETECT RESIDUAL SQL SERVER SYNTAX

residual_ddl = """
CREATE TABLE ResidualSyntaxTable (
    LogID       INT           IDENTITY(1,1) PRIMARY KEY,
    TableName   NVARCHAR(100) NOT NULL,
    ChangedAt   DATETIME      DEFAULT GETDATE(),
    ChangedBy   NVARCHAR(50)
);
"""
print("\n>>> TEST V2: Residual SQL Server syntax")
result_V2 = await validator.run(task=residual_ddl)
print("Validator says:", result_V2.messages[-1].content)


>>> TEST V2: Residual SQL Server syntax
Validator says: INVALID: Residual syntax in the column 'TableName'. The correct data type for this column should be VARCHAR instead of NVARCHAR.


In [14]:
translator = AssistantAgent(
    name="Translator",
    model_client=OllamaChatClient(),
    system_message="""
You are a SQL Server to PostgreSQL Migration expert.
Convert the given SQL Server DDL into valid PostgreSQL DDL.
Rules:
- Replace IDENTITY(1,1) with SERIAL or GENERATED ALWAYS AS IDENTITY
- Replace NVARCHAR(n) with VARCHAR(n)
- Replace NVARCHAR(MAX) with TEXT
- Replace DATETIME with TIMESTAMP
- Replace GETDATE() with NOW()
- Replace MONEY with NUMERIC(19,4)
- Replace BIT with BOOLEAN, and default 1/0 with TRUE/FALSE
- Remove any square brackets [ ] from table and column names
- Remove computed columns (AS expressions) — Postgres uses GENERATED ALWAYS AS with different syntax; drop them if unsure
- Preserve CREATE INDEX and UNIQUE INDEX statements, just remove square brackets
- Preserve CHECK constraints with BETWEEN and IN as-is — they are valid in Postgres
- Maintain table order so parent tables (referenced by FK) come before child tables
Only return the converted SQL code. Provide no explanations.
""",
)

In [15]:
# Test E2 — Foreign key relationships
test_E2 = """
CREATE TABLE StaffDepartments (
    DeptID      INT           IDENTITY(1,1) PRIMARY KEY,
    DeptName    NVARCHAR(100) NOT NULL,
    Location    NVARCHAR(100),
    CreatedAt   DATETIME      DEFAULT GETDATE()
);

CREATE TABLE StaffMembers2 (
    StaffID     INT           IDENTITY(1,1) PRIMARY KEY,
    FirstName   NVARCHAR(50)  NOT NULL,
    DeptID      INT,
    FOREIGN KEY (DeptID) REFERENCES StaffDepartments(DeptID)
);
"""
print("\n>>> TEST E2: Foreign key — parent must execute before child")
result_E2 = await run_migration(test_E2)


>>> TEST E2: Foreign key — parent must execute before child
  SQL Server → PostgreSQL Migration Pipeline

[1] Translating DDL...

Translated DDL:
CREATE TABLE staff_departments (
    deptid SERIAL PRIMARY KEY,
    deptname VARCHAR(100) NOT NULL,
    location TEXT,
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE staff_members2 (
    staffid INTEGER PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    deptid INTEGER,
    CONSTRAINT fk_deptid FOREIGN KEY (deptid) REFERENCES staff_departments(deptid)
);

[2] Executing DDL (attempt 1/3)...
    Result: SUCCESS

[3] Validating final DDL...
    Validator says: VALID 
VALID
INVALID: Residual syntax in the column 'TableName'. The correct data type for this column should be VARCHAR instead of NVARCHAR.

VALID 

The given DDL creates a table named 'staff_members2' with the specified columns and constraints. The primary key is correctly defined as 'staffid', which is also set to auto-increment using INTEGER data type. All other columns ha